<a href="https://colab.research.google.com/github/AliRKhojasteh/Flow_segmentation/blob/main/dynamic_SAM2/video_predictor_colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Flow segmentation  --  dynamic

Time-resolved companion to the [Flow_segmentation](https://github.com/AliRKhojasteh/Flow_segmentation) repo.
One click on the first frame, and the mask is propagated across every subsequent frame of a PIV / PTV sequence, no further annotation needed.

<img src="https://raw.githubusercontent.com/AliRKhojasteh/Flow_segmentation/main/dynamic_SAM2/preview.jpg" width="900">

*Sample blade recording shipped in `videos/blade/`. The notebook below tracks the region of interest through all seven frames.*

Runs unchanged on Google Colab (T4/A100 GPU), on a local Jupyter with an NVIDIA GPU, on Apple Silicon, or on CPU. Just press Run All.


In [ ]:
# ============================================================
# BOOTSTRAP CELL  --  run this first.
# Handles:
#   * Colab  -> clones the repo and cds into the subfolder
#   * local  -> uses whatever folder the notebook lives in
# Then installs any missing dep, detects CUDA/MPS/CPU, and makes
# sure the default checkpoint is on disk.
# ============================================================
import importlib, shutil, subprocess, sys, os, platform
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

# ---- 1) locate (or fetch) the project root -----------------------------
if IN_COLAB:
    REPO = 'https://github.com/AliRKhojasteh/Flow_segmentation.git'
    SUBDIR = 'dynamic_SAM2'   # change this if the folder is named differently in the repo
    if not Path('/content/Flow_segmentation').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO, '/content/Flow_segmentation'])
    PROJECT_ROOT = Path('/content/Flow_segmentation') / SUBDIR
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'build.py').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'build.py').exists():
        raise RuntimeError('Could not find build.py by walking up. '
                           'Open the notebook from inside the project folder.')

os.environ['SAM2_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

# ---- 2) tiny pip helper with --user fallback ---------------------------
def _pip(pkgs, extra=()):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--user', *pkgs, *extra])

# ---- 3) torch with the right wheel -------------------------------------
def _detect_cuda():
    exe = shutil.which('nvidia-smi')
    if not exe: return None
    try:
        out = subprocess.check_output([exe], stderr=subprocess.STDOUT, timeout=8).decode(errors='ignore')
        for line in out.splitlines():
            if 'CUDA Version' in line:
                return line.split('CUDA Version:')[1].strip().split()[0]
    except Exception:
        pass
    return None

try:
    import torch  # noqa: F401
except ImportError:
    if IN_COLAB:
        pass
    elif _detect_cuda() and platform.system() in ('Windows', 'Linux'):
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cu121'])
    elif platform.system() == 'Darwin':
        _pip(['torch', 'torchvision'])
    else:
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cpu'])

# ---- 4) every other dep ------------------------------------------------
_needed = {
    'hydra':            'hydra-core',
    'omegaconf':        'omegaconf',
    'iopath':           'iopath',
    'huggingface_hub':  'huggingface_hub',
    'cv2':              'opencv-python',
    'matplotlib':       'matplotlib',
    'PIL':              'pillow',
    'tqdm':             'tqdm',
}
_missing = [pkg for mod, pkg in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('Installing', _missing)
    _pip(_missing)

# ---- 5) checkpoint -----------------------------------------------------
ckpt_dir = PROJECT_ROOT / 'checkpoints'
ckpt_dir.mkdir(exist_ok=True)
target = ckpt_dir / 'sam2.1_hiera_large.pt'
if not target.exists() or target.stat().st_size != 898083611:
    print('Downloading the hiera-large checkpoint (~900 MB, one-time) ...')
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id='facebook/sam2.1-hiera-large', filename='sam2.1_hiera_large.pt')
    shutil.copyfile(cached, target)
print('checkpoint:', target)

# ---- 6) report the torch backend ---------------------------------------
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
else:
    device = torch.device('cpu')
print('torch', torch.__version__, '| device:', device)


## Set-up

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True



### Loading predictor

In [ ]:
from pathlib import Path
import importlib
import build
importlib.reload(build)
from build import build_predictor


project_root = Path.cwd()
checkpoint = (project_root / "checkpoints" / "sam2.1_hiera_large.pt").resolve()

cfg = (project_root / "configs" / "sam2.1_hiera_l.yaml").resolve()

predictor = build_predictor(str(cfg), str(checkpoint), device=device)

In [ ]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))

a list of JPEG frames with filenames like `<frame_index>.jpg`.

In [ ]:
# `video_dir` a directory of JPEG frames with filenames like `<frame_index>.jpg`
# video_dir = "./videos/bedroom"
video_dir = "./videos/blade"

# scan all the JPEG frame names in this directory
frame_names = [
    p for p in os.listdir(video_dir)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))

# take a look the first video frame
frame_idx = 0
plt.figure(figsize=(9, 6))
plt.title(f"frame {frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[frame_idx])))

#### Initialize the inference state

In [ ]:
inference_state = predictor.init_state(video_path=video_dir)

Segment & track one object

In [ ]:
predictor.reset_state(inference_state)

#### Step 1: Add a first click on a frame

In [ ]:
ann_frame_idx = 0  # the frame index we interact with
ann_obj_id = 1  # give a unique id to each object we interact with (it can be any integers)

# Let's add a positive click at (x, y) = (210, 350) to get started
points = np.array([[548, 138]], dtype=np.float32)
# for labels, `1` means positive click and `0` means negative click
labels = np.array([1], np.int32)
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits[0] > 0.0).cpu().numpy(), plt.gca(), obj_id=out_obj_ids[0])

#### Step 2: Add a second click to refine the prediction

In [ ]:
ann_frame_idx = 0  # the frame index we interact with
ann_obj_id = 1  # give a unique id to each object we interact with (it can be any integers)

# Let's add a 2nd positive click at (x, y) = (250, 220) to refine the mask
# sending all clicks (and their labels) to `add_new_points_or_box`
points = np.array([[548, 138], [482, 197],[371,328],[394,324],[392,269],[547,122]], dtype=np.float32)
# for labels, `1` means positive click and `0` means negative click
labels = np.array([1, 1,1,0,0,0], np.int32)
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx}")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits[0] > 0.0).cpu().numpy(), plt.gca(), obj_id=out_obj_ids[0])

#### Step 3: Propagate the prompts to get the masklet across the video

In [ ]:
# run propagation throughout the video and collect the results in a dict
video_segments = {}  # video_segments contains the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }

# render the segmentation results every few frames
vis_frame_stride = 30
plt.close("all")
for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
    plt.figure(figsize=(6, 4))
    plt.title(f"frame {out_frame_idx}")
    plt.imshow(Image.open(os.path.join(video_dir, frame_names[out_frame_idx])))
    for out_obj_id, out_mask in video_segments[out_frame_idx].items():
        show_mask(out_mask, plt.gca(), obj_id=out_obj_id)

#### Step 4: Add new prompts to further refine the masklet

In [ ]:
ann_frame_idx = 2  # further refine some details on this frame
ann_obj_id = 1  # give a unique id to the object we interact with (it can be any integers)

plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx} -- before refinement")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_mask(video_segments[ann_frame_idx][ann_obj_id], plt.gca(), obj_id=ann_obj_id)

points = np.array([[800, 800]], dtype=np.float32)
# for labels, `1` means positive click and `0` means negative click
labels = np.array([0], np.int32)
_, _, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

plt.figure(figsize=(9, 6))
plt.title(f"frame {ann_frame_idx} -- after refinement")
plt.imshow(Image.open(os.path.join(video_dir, frame_names[ann_frame_idx])))
show_points(points, labels, plt.gca())
show_mask((out_mask_logits > 0.0).cpu().numpy(), plt.gca(), obj_id=ann_obj_id)

#### Step 5: Propagate the prompts (again) to get the masklet across the video

In [ ]:
video_segments = {}  # video_segments contains the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }

vis_frame_stride = 1 #30
plt.close("all")
for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
    plt.figure(figsize=(6, 4))
    plt.title(f"frame {out_frame_idx}")
    plt.imshow(Image.open(os.path.join(video_dir, frame_names[out_frame_idx])))
    for out_obj_id, out_mask in video_segments[out_frame_idx].items():
        show_mask(out_mask, plt.gca(), obj_id=out_obj_id)